# Stage 9A: independent residual TCN

This standalone Colab notebook trains a new lightweight TCN against the verified ravaghi OOF residual. Features are rebuilt only from competition inputs, the fixed base prediction, and prefix-derived diagnostics. Stage 9A performs standard five-fold cross-fit only; it never submits to Kaggle.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
data_dir = drive_root / 'data'
artifact_dir = drive_root / 'artifacts'
assert (data_dir / 'train').is_dir(), f'Missing: {data_dir / "train"}'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
python = repo_dir / '.venv/bin/python'
subprocess.run(['uv', 'pip', 'install', '--python', str(python), 'kagglehub', 'lightgbm', 'catboost'], check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)


## Strong-base preparation

The notebook reuses the verified ravaghi `base_oof.parquet` from Stage 7 when present. If it is missing, the following cells reconstruct it from the public artifact. This fallback is memory-heavy but makes Stage 8 standalone.


In [ ]:
BASE_RUN_ID = 'stage7_public_residual_gate_full_v001'
base_run = artifact_dir / BASE_RUN_ID
if not (base_run / 'base_oof.parquet').is_file():
    download = subprocess.run([
        str(python), '-c',
        "import kagglehub; print(kagglehub.dataset_download('ravaghi/wellbore-geology-prediction-artifacts'))"
    ], check=True, capture_output=True, text=True)
    download_root = Path(download.stdout.strip().splitlines()[-1])
    train_candidates = list(download_root.rglob('data/train.csv'))
    assert len(train_candidates) == 1, train_candidates
    public_artifacts_dir = train_candidates[0].parent.parent
    probe = subprocess.run([str(python), '-c', 'import koolbox'], capture_output=True)
    if probe.returncode != 0:
        direct = subprocess.run(['uv', 'pip', 'install', '--python', str(python), 'koolbox==0.1.3'])
        if direct.returncode != 0:
            kb_download = subprocess.run([
                str(python), '-c',
                "import kagglehub; print(kagglehub.dataset_download('phongnguyn23021656/koolbox-offline'))"
            ], check=True, capture_output=True, text=True)
            kb_root = Path(kb_download.stdout.strip().splitlines()[-1])
            for wheel in kb_root.rglob('*.whl'):
                subprocess.run(['uv', 'pip', 'install', '--python', str(python), '--no-deps', str(wheel)], check=True)
    subprocess.run([
        'uv', 'run', 'rogii-public-oof',
        '--config', 'configs/experiment/public_residual_gate.yaml',
        '--public-artifacts-dir', str(public_artifacts_dir),
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
        '--run-id', BASE_RUN_ID,
    ], cwd=repo_dir, check=True)
else:
    print('Reusing strong-base OOF:', base_run)


## Run Stage 8A

Keep `LIMIT_WELLS = None` for the promotion decision. Use 10 or more only for a wiring smoke test.


In [ ]:
RUN_ID = 'stage8_safe_cutback_gate_full_v002'
LIMIT_WELLS = None
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    command = [
        'uv', 'run', 'rogii-safe-cutback',
        '--config', 'configs/experiment/stage8_safe_cutback_gate.yaml',
        '--base-run', str(base_run),
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID,
    ]
    if LIMIT_WELLS is not None:
        command += ['--limit-wells', str(LIMIT_WELLS)]
    log_path = artifact_dir / f'{RUN_ID}_driver.log'
    with log_path.open('w', encoding='utf-8') as log_handle:
        process = subprocess.Popen(
            command,
            cwd=repo_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end='')
            log_handle.write(line)
            log_handle.flush()
        return_code = process.wait()
    if return_code != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-80:]
        print('\n'.join(tail))
        raise RuntimeError(f'Stage 8 failed with exit code {return_code}. Full log: {log_path}')
else:
    print('Reusing completed run:', run_dir)


## GPU check

Use a T4 or P100 runtime. T4 is preferred for mixed-precision Conv1d training. The Colab kernel's preinstalled PyTorch is used directly; it is not downloaded into the uv environment.


In [ ]:
import importlib.util, sys
required = ['torch', 'pandas', 'pyarrow', 'sklearn', 'yaml']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyarrow', 'scikit-learn', 'pyyaml'], check=True)
import torch
assert torch.cuda.is_available(), 'Change the Colab runtime to a T4 or P100 GPU and reconnect.'
print('Python:', sys.executable)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))


## Train five cross-fit TCN models

Keep `LIMIT_WELLS = None` for the decision run. Checkpoints and logs are written to Google Drive after every fold.


In [ ]:
RUN_ID = 'stage9_residual_tcn_full_v001'
LIMIT_WELLS = None
cutback_run = artifact_dir / 'stage8_safe_cutback_gate_full_v002'
assert (cutback_run / 'candidate_matrix.parquet').is_file(), cutback_run
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    command = [
        sys.executable, '-m', 'rogii.cli.sequence',
        '--config', 'configs/experiment/stage9_residual_tcn.yaml',
        '--cutback-run', str(cutback_run),
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID,
    ]
    if LIMIT_WELLS is not None:
        command += ['--limit-wells', str(LIMIT_WELLS)]
    sequence_env = os.environ.copy()
    sequence_env['PYTHONPATH'] = str(repo_dir / 'src') + ':' + sequence_env.get('PYTHONPATH', '')
    log_path = artifact_dir / f'{RUN_ID}_driver.log'
    with log_path.open('w', encoding='utf-8') as log_handle:
        process = subprocess.Popen(
            command, cwd=repo_dir, env=sequence_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end='')
            log_handle.write(line); log_handle.flush()
        return_code = process.wait()
    if return_code != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-100:]
        print('\n'.join(tail))
        raise RuntimeError(f'Stage 9 failed with exit code {return_code}. Full log: {log_path}')
else:
    print('Reusing completed run:', run_dir)


In [ ]:
gate = json.loads((run_dir / 'gate_summary.json').read_text())
{
    'promoted_to_spatial_audit': gate['promoted_to_spatial_audit'],
    'base_rmse': gate['base_metrics']['pooled_rmse'],
    'nested_candidate_rmse': gate['candidate_metrics']['pooled_rmse'],
    'rmse_delta': gate['pooled_rmse_delta'],
    'bootstrap_95pct': [gate['bootstrap']['ci_2_5'], gate['bootstrap']['ci_97_5']],
    'improved_folds': f"{gate['improved_folds']}/{len(gate['fold_deltas'])}",
    'fold_deltas': gate['fold_deltas'],
    'gates': gate['gates'],
    'inference_weight': gate['inference_weight'],
    'selections': gate['selections'],
    'weight_report': gate['weight_report'],
    'promotion_note': gate['promotion_note'],
}


## Decision

`promoted_to_spatial_audit: True` authorizes Stage 9B spatial cross-fit, not Kaggle submission. A false result closes this TCN design without spending a Kaggle run.
